# 冻结 Stage1 f696/b472 开发期现金空间来源核验

从 f696 Recorder 读取2016–2022逐年向前分配预测，与 b472 动作训练 Recorder 中同日期决策前风险上限逐行核对。沿用固定 +0.10 SOXX / -0.10 现金预检规则，不读取未来标签列、不拟合、不运行账户。最终仍以完整 Stage1 父方案0bp费用后SOXX账户为硬门。

In [1]:
import hashlib,json,os,pathlib,shlex,subprocess,sys,tempfile
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_f696_b472_oof_cash_room_support_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists();src=plan['sources']
a_card=root/src['allocation_card'];b_card=root/src['action_card'];assert sha(a_card/'summary.json')==src['allocation_summary_sha256'] and sha(b_card/'summary.json')==src['action_summary_sha256'] and sha(root/'research/stage1_raw_intent_cash_recycle_support_20261023/summary.json')==src['prior_scaffold_support_summary_sha256'];assert sha(root/'src/workflow/intraday_recovery_direct_allocation.py')==src['projector_module_sha256']
a_summary=json.loads((a_card/'summary.json').read_text());b_summary=json.loads((b_card/'summary.json').read_text());assert a_summary['recorder_id']==src['allocation_recorder_id'] and b_summary['recorder_id']==src['action_recorder_id']
a_relative='artifacts/pit_constituent_auxiliary_training_allocation_retrained_20260928/predictions.csv';b_relative='artifacts/stage1_open_action_value_net_fees_20260928/action_value_training_frame.csv';a_path=a_card/'qlib_recorder'/src['allocation_experiment_id']/src['allocation_recorder_id']/a_relative;b_path=b_card/'qlib_recorder'/src['action_experiment_id']/src['action_recorder_id']/b_relative
for p,expected in [(a_path,src['allocation_predictions_artifact_sha256']),(b_path,src['action_training_frame_artifact_sha256'])]:
 assert p.exists() and sha(p)==expected and p.stat().st_size>0
 rel=str(p.relative_to(root));assert subprocess.run(['git','ls-files','--error-unmatch',rel],cwd=root,capture_output=True).returncode==0
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';provider=(root/src['Qlib_provider_uri']).resolve();a_uri=(a_card/'qlib_recorder').resolve();b_uri=(b_card/'qlib_recorder').resolve();qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(a_uri),'default_exp_name':a_card.name}})
with tempfile.TemporaryDirectory(prefix='stage1-parent-oof-source-') as tmp:
 a_rec=R.get_recorder(experiment_id=src['allocation_experiment_id'],recorder_id=src['allocation_recorder_id']);assert a_rec.status=='FINISHED';a_download=pathlib.Path(a_rec.download_artifact('pit_constituent_auxiliary_training_allocation_retrained_20260928/predictions.csv',tmp));assert sha(a_download)==sha(a_path)
 with R.uri_context(str(b_uri)):
  b_rec=R.get_recorder(experiment_id=src['action_experiment_id'],recorder_id=src['action_recorder_id']);assert b_rec.status=='FINISHED';b_download=pathlib.Path(b_rec.download_artifact('stage1_open_action_value_net_fees_20260928/action_value_training_frame.csv',tmp));assert sha(b_download)==sha(b_path)
  a=pd.read_csv(a_download,usecols=['signal_date','segment','soxl_weight','soxx_weight','cash_weight']);b=pd.read_csv(b_download,usecols=['signal_date','year','raw_soxl_weight','raw_soxx_weight','raw_cash_weight','current_beta_cap','target_soxl_weight','target_soxx_weight','target_cash_weight','decision_at'])
a=a[a.segment.eq('development_walk_forward')].copy();assert len(a)==len(b)==1756 and a.signal_date.is_unique and b.signal_date.is_unique;m=a.merge(b,on='signal_date',validate='one_to_one',how='outer',indicator=True);assert m._merge.eq('both').all() and len(m)==1756
for x,y in [('soxl_weight','raw_soxl_weight'),('soxx_weight','raw_soxx_weight'),('cash_weight','raw_cash_weight')]:assert np.max(np.abs(m[x]-m[y]))==0
cal=[str(x)[:10] for x in D.calendar(start_time='2016-01-04',end_time='2022-12-21',freq='day')];assert set(m.signal_date).issubset(set(cal)) and m.signal_date.is_monotonic_increasing
state={'status':'source_support_running','plan_sha256':sha(card/'plan.json'),'Qlib_Recorder_artifact_downloads':2,'Qlib_D_calendar_calls':1,'future_label_columns_loaded':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rows=[];maxerr=0.
for x in m.itertuples(index=False):
 sx=float(x.raw_soxl_weight);xx=float(x.raw_soxx_weight);cash=float(x.raw_cash_weight);cap=float(x.current_beta_cap);saved=np.array([float(x.target_soxl_weight),float(x.target_soxx_weight),float(x.target_cash_weight)]);assert np.all(np.isfinite([sx,xx,cash,cap])) and min(sx,xx,cash)>=-1e-9 and abs(sx+xx+cash-1)<1e-7
 control=project_legal_weights(sx,xx,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30);err=float(np.max(np.abs(control-saved)));maxerr=max(maxerr,err);assert err<1e-8,(x.signal_date,err)
 rawbeta=3*sx+xx;source_ok=bool(cash>=.25-1e-10 and rawbeta+.10<=cap+1e-10)
 if source_ok:
  candidate=project_legal_weights(sx,xx+.10,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30);legal=bool(np.max(np.abs(candidate-(saved+np.array([0.,.10,-.10]))))<1e-9)
 else:legal=False
 rows.append({'signal_date':x.signal_date,'year':int(x.year),'raw_cash_weight':cash,'raw_beta':rawbeta,'current_beta_cap':cap,'source_cash_ge_025':bool(cash>=.25-1e-10),'source_beta_slack_ge_010':bool(rawbeta+.10<=cap+1e-10),'source_ok':source_ok,'projected_exact_plus_010_SOXX':legal})
table=pd.DataFrame(rows);annual=[{'year':int(y),'rows':len(g),'source_ok':int(g.source_ok.sum()),'projected_exact_plus_010_SOXX':int(g.projected_exact_plus_010_SOXX.sum())} for y,g in table.groupby('year')];n=int(table.projected_exact_plus_010_SOXX.sum());years20=sum(z['projected_exact_plus_010_SOXX']>=20 for z in annual);gate=n>=100 and years20>=3
result={'decision':'fixed_f696_b472_development_source_support_failed' if not gate else 'fixed_f696_b472_development_source_support_passed','plan_sha256':sha(card/'plan.json'),'source_allocation_recorder_id':a_rec.id,'source_action_recorder_id':b_rec.id,'sessions':len(table),'exact_raw_allocation_match_rows':len(table),'max_saved_target_projection_error':maxerr,'raw_cash_ge_025':int(table.source_cash_ge_025.sum()),'beta_slack_ge_010':int(table.source_beta_slack_ge_010.sum()),'source_ok':int(table.source_ok.sum()),'projected_exact_plus_010_SOXX':n,'years_with_at_least20':years20,'support_gate_pass':gate,'annual':annual,'source_legacy_recorder_NAS_scoped_restore_verified':False,'source_CSV_Git_tracked_and_SHA_verified':True,'future_label_columns_loaded':0,'model_fits':0,'native_accounts':0,'limitation':'This is a walk-forward allocation and predecision risk-state source audit, not an OOF b472 action decision, funded whole-share settlement, local outcome label or closed-loop candidate account.'};(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
with R.uri_context(str(card/'qlib_recorder')):
 with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
  out=R.get_recorder();out.save_objects(artifact_path='inputs',plan=plan,source_artifact_hashes={'allocation_predictions':sha(a_path),'action_training_frame':sha(b_path)});out.save_objects(artifact_path='analysis',source_support=table,result=result);out.log_metrics(source_rows=len(table),source_ok=result['source_ok'],projected_eligible=n,support_gate_pass=int(gate),model_fits=0,native_accounts=0);rid,eid=out.id,str(out.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_ids':[a_rec.id,b_rec.id],'source_CSV_Git_tracked_SHA_verified':True,'source_legacy_full_Recorder_NAS_scoped_restore_verified':False,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'remote_hash_verified':True},'projected_eligible':n,'support_gate_pass':gate,'future_label_columns_loaded':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/source_support','analysis/result']},'limitations':['Legacy source CSVs are Git tracked with exact hashes, but whole old Recorder NAS receipts were not found; preserve local objects.','No OOF b472 action decision or funded execution was tested.','No local profit label, full Stage1 0bp SOXX account or independent forward evidence.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':result['decision'],'source_ok':result['source_ok'],'legal':n,'annual':annual,'recorder_id':rid},ensure_ascii=False))


2026/10/23 14:17:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[5914:MainThread](2026-10-23 14:17:48,887) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[5914:MainThread](2026-10-23 14:17:48,889) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[5914:MainThread](2026-10-23 14:17:48,890) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[5914:MainThread](2026-10-23 14:17:48,955) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_f696_b472_oof_cash_room_support_20261023.


[5914:MainThread](2026-10-23 14:17:48,957) INFO - qlib.workflow - [exp.py:258] - Experiment 879769726506886081 starts running ...


[5914:MainThread](2026-10-23 14:17:49,033) INFO - qlib.workflow - [recorder.py:345] - Recorder 45c6708c35f2421eaeca75d66cbce6b5 starts running under Experiment 879769726506886081 ...


[5914:MainThread](2026-10-23 14:17:49,966) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "fixed_f696_b472_development_source_support_failed", "source_ok": 25, "legal": 25, "annual": [{"year": 2016, "rows": 252, "source_ok": 0, "projected_exact_plus_010_SOXX": 0}, {"year": 2017, "rows": 251, "source_ok": 23, "projected_exact_plus_010_SOXX": 23}, {"year": 2018, "rows": 251, "source_ok": 2, "projected_exact_plus_010_SOXX": 2}, {"year": 2019, "rows": 252, "source_ok": 0, "projected_exact_plus_010_SOXX": 0}, {"year": 2020, "rows": 253, "source_ok": 0, "projected_exact_plus_010_SOXX": 0}, {"year": 2021, "rows": 252, "source_ok": 0, "projected_exact_plus_010_SOXX": 0}, {"year": 2022, "rows": 245, "source_ok": 0, "projected_exact_plus_010_SOXX": 0}], "recorder_id": "45c6708c35f2421eaeca75d66cbce6b5"}


# 结论：冻结 f696/b472 开发期现金空间来源核验

本卡纠正了上一份0bp训练支架的来源边界。旧支架明确是“开盘总执行、无恢复动作”，不是完整冻结 Stage1。这里改用 **f696 Recorder 的2016–2022逐年向前配置预测**，并以 **b472 Recorder 同日期动作训练框架的决策前 beta 上限**核对。两份记录的原始 SOXL/SOXX/现金权重 **1756/1756 日完全相同**，保存目标与原投影最大差 `2.22e-16`。只读取决策前列，不读取未来标签列、不拟合、不运行账户。

沿用事先冻结的“保持自身 beta 上限，+10% SOXX / −10%现金”动作：**1092** 日原始现金足够，只有 **25** 日 beta 空间也足够并可精确投影。其中 **2017 年23日、2018年2日；2016及2019–2022均为0**。未达到总数100且至少3年各20日的来源支持门，这一固定版本在收益标签之前关闭。新 Qlib Recorder `45c6708c35f2421eaeca75d66cbce6b5` 的选定表 NAS 冷读回一致；两份旧 f696/b472 Recorder 也补做了逐件 NAS 冷恢复，原始 Git 工件未改。

这说明在开发期的冻结配置家族下，**现金并不能简单转换成额外 SOXX 而保持风险上限不变**。完整2023–2026 Stage1账户另有536个原始意图受限且名义 beta 不绑定的 EXECUTE 日；该来源分布不同，不能把它们直接当作训练样本。此次没有核验 b472 逐年向前动作决定、整股成交、局部净收益或完整自身状态账户；SOXX 0bp、2/5bp及独立前向门均未通过。下一机制必须有真实决策时点信息与跨年份训练支持，不能只缩小这条固定动作或选2017年。
